# Inference and Attention

This notebook runs autoregressive English-to-French translation and inspects token probabilities and attention patterns.

In [ ]:
import torch
import matplotlib.pyplot as plt
from config import get_config
from translate import load_translation_model, encode_source, greedy_decode, translate_text

In [ ]:
config = get_config()
model, tokenizer_src, tokenizer_tgt, device = load_translation_model(config)
device

In [ ]:
sentences = [
    'Hello, how are you?',
    'The weather is beautiful today.',
    'I am learning about neural networks.',
    'This model translates English into French.'
]
translations = [(sentence, translate_text(sentence, model, tokenizer_src, tokenizer_tgt, config, device)) for sentence in sentences]
translations

In [ ]:
text = sentences[0]
source, source_mask = encode_source(text, tokenizer_src, config['seq_len'])
source = source.to(device)
source_mask = source_mask.to(device)
with torch.inference_mode():
    encoder_output = model.encode(source, source_mask)
    decoder_input = torch.tensor([[tokenizer_tgt.token_to_id('[SOS]')]], device=device)
    steps = []
    while decoder_input.size(1) < config['seq_len']:
        decoder_mask = torch.tril(torch.ones(decoder_input.size(1), decoder_input.size(1), dtype=torch.bool, device=device)).unsqueeze(0)
        decoder_output = model.decode(encoder_output, source_mask, decoder_input, decoder_mask)
        logits = model.project(decoder_output[:, -1])
        probabilities = torch.softmax(logits, dim=-1)
        values, indices = torch.topk(probabilities, k=min(8, probabilities.size(-1)), dim=-1)
        steps.append((values.squeeze(0).cpu(), indices.squeeze(0).cpu()))
        next_token = indices[:, :1]
        decoder_input = torch.cat([decoder_input, next_token], dim=1)
        if next_token.item() == tokenizer_tgt.token_to_id('[EOS]'):
            break
len(steps)

In [ ]:
confidence = [values[0].item() for values, _ in steps]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(confidence) + 1), confidence, marker='o')
ax.set_xlabel('Decoding step')
ax.set_ylabel('Top-token probability')
ax.set_title('Autoregressive Token Confidence')
ax.grid(alpha=0.25)
fig.tight_layout()

In [ ]:
source_tokens = tokenizer_src.encode(text).tokens
generated_tokens = tokenizer_tgt.decode(decoder_input.squeeze(0).cpu().numpy(), skip_special_tokens=False)
source_tokens, generated_tokens